# Grid Search v3 — all five metrics required — Macro F1 targets, nested dataset validation

This notebook is self-contained. It audits and updates `result1.csv`, then runs
nested leave-one-dataset-out selection, following the original signed linear grid:
step 0.1, sum of absolute coefficients 1, each absolute coefficient at least 0.1.
All five metrics are mandatory. Only standardization is selected in inner folds.

Target: observed **MacroF1**. The search optimizes Spearman ordering of these F1
values; scores are ranking scores, not calibrated F1 predictions. Two absolute-gap
features mean `abs(MeanGap)` and `abs(EtaGap)` as requested. Neither is the mean of
absolute bootstrap gaps. The five search inputs remain the original five metrics.

Outer held-out datasets never select coefficients, normalization, or feature budget.
Inner dataset folds select zscore/robust/rank and all five nonzero metrics.
Unlabeled within-target-dataset scaling matches deployment. All seven datasets were
previously inspected, so these are exploratory results, not an untouched final test.

This experiment does not select single/two-metric alternatives. Previous experiments remain unchanged.


In [1]:
from pathlib import Path
from itertools import product
import json, re, shutil
import numpy as np
import pandas as pd
from scipy.stats import rankdata, kendalltau
from IPython.display import display

BASE = Path.cwd()
if (BASE / "GridSearch" / "result1.csv").exists():
    BASE = BASE / "GridSearch"
ROOT = BASE.parent
OUT = BASE / "results_nested_five_required"
OUT.mkdir(exist_ok=True)
METRICS = ["SameMean", "DffMean", "SameAbsEta", "DiffAbsEta", "Chi2Distance"]
METHODS = ["zscore", "robust", "rank"]
BUDGETS = {"five_required": 5}
MODEL_NAMES = {
    "google-bert/bert-base-uncased": "BERT",
    "allenai/scibert_scivocab_uncased": "SciBERT",
    "nlpaueb/legal-bert-base-uncased": "LegalBERT",
    "ahmedrachid/FinancialBERT": "FinancialBERT",
    "Lianglab/PharmBERT-uncased": "PharmBERT",
    "recobo/agriculture-bert-uncased": "AgricultureBERT",
    "recobo/chemical-bert-uncased": "ChemicalBERT",
}


In [2]:
# Audit F1 from actual source records, never reconstruct F1 from ranks.
data = pd.read_csv(BASE / "result1.csv")
backup = BASE / "result1_before_final_audit.csv"
if not backup.exists():
    shutil.copy2(BASE / "result1.csv", backup)
source_info = []
f1_sources = {}
for dataset, source, value, unit in [
    ("Ohsumed", "Ohsumed/results/ohsumed_5fold/ohsumed_5fold_results.csv", "macro_f1", "fold"),
    ("ACGNews", "ACGNews/results/ag_news_5fold/ag_news_5fold_results.csv", "macro_f1", "fold"),
    ("arXiv", "arXiv/results/arxiv_9class_5fold/arxiv_9class_5fold_results.csv", "macro_f1", "fold"),
    ("Rheology", "Rheology/results/rheology_5fold/rheology_5fold_results.csv", "macro_f1", "fold"),
    ("ECtHR", "ECtHR/real_ranking_raw_results.csv", "test_macro_f1", "seed"),
]:
    raw = pd.read_csv(ROOT / source)
    assert not raw.duplicated(["model", unit]).any()
    assert raw.groupby("model")[unit].nunique().eq(3 if unit == "seed" else 5).all()
    f1_sources[dataset] = raw.groupby("model")[value].mean()
    source_info.append({"Dataset": dataset, "Source": source,
                        "Aggregation": "3 seeds" if unit == "seed" else "5 folds",
                        "Precision": "saved CSV precision"})

for dataset, source, cell_index in [
    ("FinancialData", "AgusDatasetBasedCompare/financial_3epochAmy.ipynb", 14),
    ("EnvData", "fullFIneTuningResults/envData_3epochAmy.ipynb", 6),
]:
    notebook = json.loads((ROOT / source).read_text())
    text = "".join("".join(o.get("text", [])) for o in notebook["cells"][cell_index].get("outputs", []))
    alternatives = "|".join(re.escape(m) for m in MODEL_NAMES)
    model_by_row = dict(re.findall(r"^\s*(\d+)\s+(" + alternatives + r")\s+", text, re.M))
    marker = "macro_f1_mean  macro_f1_std"
    assert marker in text, source
    values = re.findall(r"^\s*(\d+)\s+([0-9]+\.[0-9]+)\s+", text.split(marker)[-1], re.M)
    found = {model_by_row[i]: float(v) for i, v in values if i in model_by_row}
    assert set(found) == set(MODEL_NAMES), source
    f1_sources[dataset] = pd.Series(found)
    source_info.append({"Dataset": dataset, "Source": source, "Aggregation": "5 folds",
                        "Precision": "printed 6 decimal places; no invented extra precision"})

inverse = {short: full for full, short in MODEL_NAMES.items()}
corrections = []
for index, row in data.iterrows():
    value = float(f1_sources[row.Dataset][inverse[row.ModelName]])
    if not np.isclose(row.MacroF1, value, rtol=0, atol=1e-12):
        corrections.append({"row": int(index), "column": "MacroF1", "old": row.MacroF1, "new": value})
    data.loc[index, "MacroF1"] = value
for absolute, signed in [("AbsMeanGap", "MeanGap"), ("AbsEtaGap", "EtaGap")]:
    data[absolute] = data[signed].abs()
assert len(data) == 49 and data.groupby("Dataset").size().eq(7).all()
assert not data.duplicated(["Dataset", "ModelName"]).any()
assert set(data.ModelName) == set(MODEL_NAMES.values())
assert data.select_dtypes("number").notna().all().all()
assert np.isfinite(data.select_dtypes("number").to_numpy()).all()
assert data.MacroF1.between(0, 1).all()
# Cross-check new prediction feature values against recovered three-run records.
feature_checks = 0
for dataset in ["ACGNews", "arXiv", "Rheology"]:
    raw = pd.read_csv(BASE / "results_nested" / f"{dataset}_recovered_repetitions.csv")
    assert len(raw) == 21 and raw.groupby("model").size().eq(3).all()
    means = raw.groupby("model")[METRICS].mean()
    for model, values in means.iterrows():
        mask = data.Dataset.eq(dataset) & data.ModelName.eq(MODEL_NAMES[model])
        assert mask.sum() == 1
        for metric in METRICS:
            old = float(data.loc[mask, metric].iloc[0]); new = float(values[metric])
            if not np.isclose(old, new, atol=1e-12, rtol=0):
                corrections.append({"Dataset": dataset, "Model": model, "column": metric, "old": old, "new": new})
                data.loc[mask, metric] = new
            feature_checks += 1
assert np.allclose(data.AbsMeanGap, data.MeanGap.abs())
assert np.allclose(data.AbsEtaGap, data.EtaGap.abs())
data.to_csv(BASE / "result1.csv", index=False)
pd.DataFrame(source_info).to_csv(OUT / "f1_sources.csv", index=False)
(OUT / "data_audit.json").write_text(json.dumps({"rows": len(data), "feature_checks": feature_checks,
    "corrections": corrections, "absolute_gap_definition": "abs(stored signed gap)"}, indent=2))
display(pd.DataFrame(source_info))
print("Audit passed:", len(data), "rows;", feature_checks, "feature checks;", len(corrections), "source-value corrections")


         Dataset                                                           Source Aggregation                                              Precision
0        Ohsumed          Ohsumed/results/ohsumed_5fold/ohsumed_5fold_results.csv     5 folds                                    saved CSV precision
1        ACGNews          ACGNews/results/ag_news_5fold/ag_news_5fold_results.csv     5 folds                                    saved CSV precision
2          arXiv  arXiv/results/arxiv_9class_5fold/arxiv_9class_5fold_results.csv     5 folds                                    saved CSV precision
3       Rheology       Rheology/results/rheology_5fold/rheology_5fold_results.csv     5 folds                                    saved CSV precision
4          ECtHR                               ECtHR/real_ranking_raw_results.csv     3 seeds                                    saved CSV precision
5  FinancialData                AgusDatasetBasedCompare/financial_3epochAmy.ipynb     5 folds  printed 6 d

In [3]:
def compositions(total, parts):
    if parts == 1:
        yield (total,)
        return
    for value in range(total + 1):
        for rest in compositions(total - value, parts - 1):
            yield (value,) + rest

STEP = 0.1
units = round(1 / STEP)
rows = []
for allocation in compositions(units, len(METRICS)):
    if any(value == 0 for value in allocation):
        continue
    weights = np.asarray(allocation, dtype=float) / units
    active = np.flatnonzero(weights)
    for signs in product([-1., 1.], repeat=len(active)):
        row = np.zeros(len(METRICS))
        row[active] = weights[active] * signs
        rows.append(row)
grid = np.asarray(rows)
active_counts = np.count_nonzero(grid, axis=1)
assert len(grid) == 4032 and np.allclose(np.abs(grid).sum(1), 1)
assert (np.abs(grid) >= 0.1 - 1e-12).all()
assert (active_counts == 5).all()


def standardize(x, method):
    if method == "rank":
        r = rankdata(x, axis=0, method="average") / len(x)
        return r - r.mean(0)
    center = x.mean(0) if method == "zscore" else np.median(x, axis=0)
    spread = x.std(0) if method == "zscore" else np.median(np.abs(x-center), axis=0) / .6745
    return np.divide(x-center, spread, out=np.zeros_like(x), where=~np.isclose(spread, 0))


def prepare(frame):
    datasets = sorted(frame.Dataset.unique())
    rho, cache = {}, {}
    for method in METHODS:
        matrix = np.zeros((len(grid), len(datasets)))
        for j, name in enumerate(datasets):
            group = frame.loc[frame.Dataset.eq(name)].sort_values("ModelName").reset_index(drop=True)
            scores = np.round(standardize(group[METRICS].to_numpy(float), method) @ grid.T, 12)
            ranks = rankdata(-scores, axis=0, method="average")
            actual = rankdata(-group.MacroF1.to_numpy(), method="average")
            p = ranks-ranks.mean(0); t = actual-actual.mean()
            denominator = np.sqrt((t*t).sum() * (p*p).sum(0))
            matrix[:, j] = np.divide(t @ p, denominator, out=np.zeros(len(grid)), where=denominator > 0)
            cache[method, j] = (group, scores, ranks, actual)
        rho[method] = matrix
    return datasets, rho, cache


def choose(rho, indices, budget):
    objective = rho[:, indices].mean(axis=1)
    allowed = np.flatnonzero(active_counts <= budget)
    maximum = objective[allowed].max()
    tied = allowed[np.isclose(objective[allowed], maximum, atol=1e-12, rtol=0)]
    return int(tied[np.argmin(active_counts[tied])])


def select_config(indices, rhos, families):
    options = []
    for family in families:
        for method in METHODS:
            scores = []
            for held in indices:
                train = [i for i in indices if i != held]
                best = choose(rhos[method], train, BUDGETS[family])
                scores.append(rhos[method][best, held])
            options.append({"Family": family, "Standardization": method, "InnerSpearman": float(np.mean(scores))})
    best = sorted(options, key=lambda x: (-round(x["InnerSpearman"], 12), BUDGETS[x["Family"]], METHODS.index(x["Standardization"])))[0]
    return best, options

names, rhos, cache = prepare(data)
print("Datasets:", names, "| Signed coefficient candidates:", len(grid))


Datasets: ['ACGNews', 'ECtHR', 'EnvData', 'FinancialData', 'Ohsumed', 'Rheology', 'arXiv'] | Signed coefficient candidates: 4032


In [4]:
fold_rows, predictions, coefficients, config_scores, baseline_rows = [], [], [], [], []
for held, name in enumerate(names):
    train = [i for i in range(len(names)) if i != held]
    selected, options = select_config(train, rhos, list(BUDGETS))
    config_scores.extend({"HeldOutDataset": name, **option} for option in options)
    for procedure in ["selected", *BUDGETS]:
        config = selected if procedure == "selected" else select_config(train, rhos, [procedure])[0]
        method = config["Standardization"]
        best = choose(rhos[method], train, BUDGETS[config["Family"]])
        group, score_matrix, rank_matrix, actual = cache[method, held]
        predicted = rank_matrix[:, best]
        score = score_matrix[:, best]
        order = np.lexsort((group.ModelName.to_numpy(), -score))
        f1 = group.MacroF1.to_numpy()
        metrics = {
            "TestSpearman": float(rhos[method][best, held]),
            "TestKendall": float(kendalltau(actual, predicted).statistic) if np.unique(predicted).size > 1 else 0.,
            "TestRankMAE": float(np.abs(actual-predicted).mean()),
            "Top1F1Regret": float(f1.max()-f1[order[0]]),
            "Top3F1Regret": float(f1.max()-f1[order[:3]].max()),
            "Top3ContainsBest": int(np.any(f1[order[:3]] == f1.max())),
        }
        fold_rows.append({"HeldOutDataset": name, "Procedure": procedure, **config, **metrics})
        predictions.append(group[["Dataset", "ModelName", "MacroF1"]].assign(
            Procedure=procedure, RealRank=actual, PredictionScore=score, PredictedRank=predicted))
        coefficients.extend({"HeldOutDataset": name, "Procedure": procedure, "Metric": metric,
                             "Coefficient": float(value)} for metric, value in zip(METRICS, grid[best]))
    # Rank-based constant model baseline, trained only on the other datasets.
    training = data.loc[data.Dataset.isin([names[i] for i in train])].copy()
    training["Rank"] = training.groupby("Dataset").MacroF1.rank(ascending=False, method="average")
    preference = training.groupby("ModelName").Rank.mean().sort_values(kind="stable")
    target = data.loc[data.Dataset.eq(name)].set_index("ModelName").MacroF1
    baseline_rows.append({"HeldOutDataset": name, "SelectedModel": preference.index[0],
        "Top1F1Regret": float(target.max()-target[preference.index[0]]),
        "Top3F1Regret": float(target.max()-target.loc[preference.index[:3]].max()),
        "Top3ContainsBest": int(target.loc[preference.index[:3]].max()==target.max())})
    print(name, selected)

fold_results = pd.DataFrame(fold_rows)
fold_results.to_csv(OUT / "outer_fold_results.csv", index=False)
pd.concat(predictions, ignore_index=True).to_csv(OUT / "outer_predictions.csv", index=False)
pd.DataFrame(coefficients).to_csv(OUT / "outer_coefficients.csv", index=False)
pd.DataFrame(config_scores).to_csv(OUT / "inner_configuration_scores.csv", index=False)
pd.DataFrame(baseline_rows).to_csv(OUT / "constant_model_baseline.csv", index=False)
columns = ["TestSpearman", "TestKendall", "TestRankMAE", "Top1F1Regret", "Top3F1Regret", "Top3ContainsBest"]
cv_summary = fold_results.groupby("Procedure")[columns].agg(["mean", "std"])
cv_summary.to_csv(OUT / "cv_summary.csv")
display(cv_summary)


ACGNews {'Family': 'five_required', 'Standardization': 'zscore', 'InnerSpearman': 0.30357142857142855}
ECtHR {'Family': 'five_required', 'Standardization': 'zscore', 'InnerSpearman': 0.6369047619047619}
EnvData {'Family': 'five_required', 'Standardization': 'rank', 'InnerSpearman': 0.3665293133622722}
FinancialData {'Family': 'five_required', 'Standardization': 'robust', 'InnerSpearman': 0.6190476190476191}
Ohsumed {'Family': 'five_required', 'Standardization': 'robust', 'InnerSpearman': 0.36904761904761907}
Rheology {'Family': 'five_required', 'Standardization': 'robust', 'InnerSpearman': 0.30952380952380953}
arXiv {'Family': 'five_required', 'Standardization': 'robust', 'InnerSpearman': 0.3333333333333333}
              TestSpearman           TestKendall           TestRankMAE           Top1F1Regret           Top3F1Regret           Top3ContainsBest         
                      mean       std        mean       std        mean       std         mean       std         mean       std   

In [5]:
# Verify held-out F1 does not participate in configuration or weight selection.
for held, name in enumerate(names):
    train = [i for i in range(len(names)) if i != held]
    before, _ = select_config(train, rhos, list(BUDGETS))
    before_index = choose(rhos[before["Standardization"]], train, BUDGETS[before["Family"]])
    changed = data.copy()
    mask = changed.Dataset.eq(name)
    changed.loc[mask, "MacroF1"] = 1-changed.loc[mask, "MacroF1"]
    _, altered_rhos, _ = prepare(changed)
    after, _ = select_config(train, altered_rhos, list(BUDGETS))
    after_index = choose(altered_rhos[after["Standardization"]], train, BUDGETS[after["Family"]])
    assert before == after and before_index == after_index
validation = {"held_out_target_reversal": "PASS on all 7 datasets", "rows": 49,
              "target": "MacroF1", "independent_final_test": False}
(OUT / "validation.json").write_text(json.dumps(validation, indent=2))
print(validation)

# Final candidate uses all seven datasets; its fit score is NOT a validation score.
config, _ = select_config(list(range(len(names))), rhos, list(BUDGETS))
index = choose(rhos[config["Standardization"]], list(range(len(names))), BUDGETS[config["Family"]])
final_formula = {**config, "Coefficients": dict(zip(METRICS, grid[index].tolist())),
                 "FullFitSpearman": float(rhos[config["Standardization"]][index].mean()),
                 "Interpretation": "ranking score, not predicted F1; full-data exploratory fit"}
(OUT / "final_formula.json").write_text(json.dumps(final_formula, indent=2))
print(json.dumps(final_formula, indent=2))


{'held_out_target_reversal': 'PASS on all 7 datasets', 'rows': 49, 'target': 'MacroF1', 'independent_final_test': False}
{
  "Family": "five_required",
  "Standardization": "zscore",
  "InnerSpearman": 0.4693877551020408,
  "Coefficients": {
    "SameMean": -0.3,
    "DffMean": 0.2,
    "SameAbsEta": 0.1,
    "DiffAbsEta": -0.3,
    "Chi2Distance": 0.1
  },
  "FullFitSpearman": 0.6020408163265305,
  "Interpretation": "ranking score, not predicted F1; full-data exploratory fit"
}


## Interpretation and limitations

Use outer test results for evaluation. Do not report full-data fit as test performance.
Family-specific outer results compare predefined pipelines; choosing a winner after
seeing those scores is additional development. Standard deviations across seven
outer folds are descriptive, not independent confidence intervals. Top-k selection
ties are broken by model name to fix the selection budget.

Macro F1 sources differ: ECtHR is a three-seed test mean; other tasks use five-fold
means. Financial and EnvData only retain six-decimal displayed mean values. Source
ranks may differ from old manually entered ranks. Upstream duplicate-text leakage,
old versus unified sampling, and model-training differences remain limitations.
No claim is made that this notebook fixes those upstream experiments. Candidate
models must match the same seven-model set. No new BERT training runs here.

This experiment does not select single/two-metric alternatives. Previous experiments remain unchanged.
